# 📈 Black-Litterman 模型 - 大类资产配置量化复现

**文献来源**: 国泰君安证券研究 | 大类资产配置量化模型研究系列之二  
**报告**: 《手把手教你实现 Black-Litterman 模型》  
**作者**: 廖静池、张雪杰 | 2023.04.05

## 0. 环境配置

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# 项目根目录
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
sys.path.insert(0, PROJECT_ROOT)

print(f"项目根目录: {PROJECT_ROOT}")
print(f"Python: {sys.version}")

## 1. 数据获取

In [ ]:
from source.data_loader import DataLoader, load_all_returns

# 方法1: 一键获取所有资产数据 (优先从本地缓存读取)
print("=" * 60)
print("Step 1: 加载大类资产日频收益率数据")
print("=" * 60)

try:
    returns = load_all_returns(
        start_date='20061101',
        end_date='20230131',
        force_reload=False  # True=强制重新拉取
    )
    print(f"\n✅ 数据加载成功: {returns.shape}")
    print(f"   日期范围: {returns.index[0].date()} ~ {returns.index[-1].date()}")
    print(f"   资产列表: {list(returns.columns)}")
except Exception as e:
    print(f"❌ 数据获取失败: {e}")
    print("提示: 检查网络或设置 force_reload=True 强制重新拉取")

In [ ]:
# 数据概览
display(returns.describe().round(4))

## 2. 资产相关性分析

In [ ]:
from source.plot import PlotEngine
import numpy as np
import pandas as pd

class DummyBT:
    def __init__(self, asset_names, daily_returns):
        self.asset_names = asset_names
        self.daily_returns = daily_returns

# 相关性热力图
dummy_bt = DummyBT(list(returns.columns), returns)
plot_eng = PlotEngine(dummy_bt, output_dir=os.path.join(PROJECT_ROOT, 'output'))
plot_eng.plot_correlation_matrix(daily_returns=returns)

## 3. Black-Litterman 模型单期演示

In [ ]:
from source.bl_model import BlackLittermanModel
import numpy as np

print("=" * 60)
print("Step 2: Black-Litterman 模型单期演示")
print("=" * 60)

# 取最近 ~60 个月日频数据
bl_data = returns[-(60 * 22):]  # 大约60个月
print(f"使用最近 {len(bl_data)} 条日频数据")

# ─── BL策略1: λ=10 固定 ───
bl_model_s1 = BlackLittermanModel(
    returns=bl_data,
    market_weights='benchmark',  # 股:债:商 = 1:8:1
    risk_aversion=10.0,
    tau=1.0/60,                   # τ = 1/(T-n)
    omega_method='default',
    rf_rate=2.0,
    view_lookback=1,              # 用过去1个月动量作为观点
)

print("\n【BL模型参数】")
print(f"  风险厌恶系数 λ = {bl_model_s1.risk_aversion}")
print(f"  观点权重 τ = {bl_model_s1.tau:.5f}")
print(f"  观点来源: 过去1个月动量收益率")
print(f"  Ω 计算方法: P*Σ*P^T (默认)")

print("\n【先验 vs 后验收益】")
summary_df = bl_model_s1.summary()
summary_df['观点收益(%)'] = bl_model_s1.Q_
display(summary_df.round(4))

In [ ]:
# 先验 vs 后验收益可视化
plot_eng.plot_prior_vs_posterior(
    prior_mu=bl_model_s1.pi_prior_,
    posterior_mu=bl_model_s1.posterior_mu_,
    asset_names=list(bl_data.columns),
    title='BL模型: 先验均衡收益 vs 后验收益 (单期示例)'
)

## 4. 完整回测

In [ ]:
from source.backtest import BacktestEngine
import config

print("=" * 60)
print("Step 3: 启动完整回测")
print("=" * 60)
print(f"回测区间: {config.BACKTEST_START} ~ {config.BACKTEST_END}")
print(f"调仓频率: 月末 (每月1次)")
print(f"股票上限: {config.STOCK_CAP*100:.0f}%  商品上限: {config.COMMODITY_CAP*100:.0f}%  换手率限制: {config.TURNOVER_LIMIT*100:.0f}%")

# 初始化回测引擎
bt_engine = BacktestEngine(
    returns=returns,
    rf_rate=config.RF_RATE,
    stock_cap=config.STOCK_CAP,
    commodity_cap=config.COMMODITY_CAP,
    turnover_limit=config.TURNOVER_LIMIT,
    rebalance_freq=config.REBALANCE_FREQ,
    lookback_months=config.LOOKBACK_MONTHS,
)

# 运行回测 (4种策略)
result = bt_engine.run(
    strategies=['BL_S1', 'BL_S2', 'MVO', 'FIXED'],
    start_date=config.BACKTEST_START,
    end_date=config.BACKTEST_END,
    risk_aversion=config.RISK_AVERSION,
    tau=1.0 / config.LOOKBACK_MONTHS,
)

## 5. 回测结果分析

In [ ]:
# 策略绩效汇总
print("\n" + "=" * 70)
print("  各策略绩效汇总")
print("=" * 70)
summary_table = result.summary_table()
display(summary_table.round(4))

In [ ]:
# 打印完整摘要
result.print_summary()

## 6. 可视化

In [ ]:
from source.plot import PlotEngine

output_dir = os.path.join(PROJECT_ROOT, 'output')
os.makedirs(output_dir, exist_ok=True)

plot_eng = PlotEngine(result, output_dir=output_dir)

print("生成图表 ...")

In [ ]:
# 图1: 累计收益曲线对比 (对标研报图4)
fig1 = plot_eng.plot_cumulative_returns(
    title='BL模型策略 vs 基准策略累计收益对比 (2012-2023)'
)

In [ ]:
# 图2: 回撤对比 (对标研报图5)
fig2 = plot_eng.plot_drawdown(
    title='BL模型策略1 vs MVO基准回撤对比'
)

In [ ]:
# 图3: BL策略1 权重配置堆叠图 (对标研报图6)
fig3 = plot_eng.plot_weights('BL_S1', title='BL策略1 资产配置权重 (月频调仓)')

In [ ]:
# 图4: MVO基准策略权重
fig4 = plot_eng.plot_weights('MVO', title='MVO均值方差基准策略权重')

In [ ]:
# 图5: 绩效指标柱状图
fig5 = plot_eng.plot_stats_bar(
    title='各策略绩效指标对比'
)

In [ ]:
# 图6: 年度收益热力图
if hasattr(result, 'yearly_stats') and result.yearly_stats:
    fig6 = plot_eng.plot_yearly_heatmap()

## 7. 数据说明与已知问题

⚠️ **数据获取重要提示**:

1. **中债指数 (CR_GOV / CR_CORP)**: 
   - akshare `bond_zh_cox_index` 和 tushare 债券指数接口可能不稳定
   - 如获取失败，请在 `source/data_loader.py` 中检查接口状态
   - **替代方案**: 可在 config.py 中设置 `USE_ASSET = ['CSI300', 'SP500', 'NHCI']` 跳过债券

2. **恒生指数 (HSI)**: 
   - yfinance 的 `^HSI` 可能缺少部分历史数据
   - fallback 方案: 使用 `2800.HK` 代替

3. **南华商品指数 (NHCI)**: 
   - akshare `futures_nh_commodity_index` 接口可能缺失
   - fallback 方案: 使用黄金期货或商品指数替代

4. **cvxopt 安装**: 
   ```bash
   pip install cvxopt
   ```
   若安装失败，程序会自动降级使用 scipy.optimize

## 8. 保存结果

In [ ]:
# 保存CSV结果
result.save_results(output_dir)
print(f"\n✅ 结果已保存至: {output_dir}")

# 列出输出文件
import os
output_files = os.listdir(output_dir)
print(f"\n输出文件 ({len(output_files)} 个):")
for f in sorted(output_files):
    fpath = os.path.join(output_dir, f)
    size = os.path.getsize(fpath)
    print(f"  {f:<45} {size/1024:.1f} KB")

---

**📌 研报关键结论**:

1. BL模型通过贝叶斯方法将**主观观点**与**市场均衡收益**结合，有效解决了MVO对预期收益敏感的缺陷
2. BL模型策略 (λ=10, 10%股+80%债+10%商品约束) 整体优于MVO和固定权重基准
3. 主观观点的预测能力是BL模型效果的关键，本报告用**一个月动量**作为观点，验证了模型有效性
4. BL模型在市场大幅波动时 (如2020、2022年) 回撤控制显著优于传统MVO